# Verify the 6 Appendix-F trivial tail of `ms640_solved` @ 1M nodes (sum cap)

`data/ms640_solved.txt` = **634** shared easy MS lines + **6** paper-trivial presentations that live
deeper in `1190MS.txt` (lines **635–640** of `ms640_solved`; **not** `1190MS` idx 634–639, which
Appendix F marks as counterexample classes). This notebook runs those six at **1,000,000** nodes under
the notebook's **sum cap** (`|r1| + |r2| < 100`) via the numba-backed verbatim solver
(`greedy_ac.py`), and replays every solve through `verify_path`.

**Why Colab, not a laptop.** Each 1M sum-cap run peaks at **~24 GB** RAM (measured: 4.9 GB @ 200k nodes,
growth ~linear — the visited dict keeps one string-tuple key per *pushed* neighbor, and under the sum cap
relators grow to ~100 so almost nothing is capped out). A 16 GB machine gets jetsam-killed around 600k
nodes. **You must use a High-RAM runtime:** Runtime → Change runtime type → **High-RAM** (~51 GB, Colab
Pro). Free Colab (~13 GB) is *not* enough — on free Colab drop `BUDGET` to ~400k for a partial run.

**Time / resumability.** ≈1 h per presentation (≈305 nodes/s measured) ⇒ ~5–6 h for all six at `WORKERS=1`.
The stream is **append-only, resumable** (JSONL on Drive), keyed by `ms640_line` (635–640).

This is a *thin wrapper* over `experiments/stable_ac/baseline_n2/greedy_ac.py` (numba JIT); no solver
logic is redefined here.

In [ ]:
# --- parameters -------------------------------------------------------------
REPO_URL  = "https://github.com/Avi161/ACSolverX.git"
BRANCH    = "test/stable-ac-moves"
DRIVE_OUT = "/content/drive/MyDrive/acsolverx_results"   # stream persists here across preemption
USE_DRIVE = True             # False -> write beside the repo instead

# ms640_solved.txt lines 635-640 (0-indexed 634-639) — Appendix-F trivial tail
LINE_IDXS = list(range(634, 640))
# 0-based line idx -> (1190MS idx, MS label) for logging only
TAIL6 = {
    634: (955,  "MS(6, 'XYYxyyy')"),
    635: (668,  "MS(6, 'yyxYYYX')"),
    636: (1132, "MS(7, 'YxyXYYY')"),
    637: (717,  "MS(7, 'yxYXYYY')"),
    638: (698,  "MS(7, 'yxYXyyy')"),
    639: (1046, "MS(7, 'yyyXYxy')"),
}
BUDGET    = 1_000_000        # nodes  (paper Table 1 GS-Sub @ 1M tier)
MAX_LEN   = 100              # sum cap:  |r1| + |r2| < MAX_LEN
CAP       = "sum"            # notebook native cap (paper repro gate)
WORKERS   = 1                # each run peaks ~24 GB; 1 fits a 51 GB High-RAM box
OUT_NAME  = "ms640_tail6_sum_L100_1M.jsonl"

In [ ]:
# --- setup: mount Drive, clone repo, install deps ---------------------------
import os, sys, subprocess, json

IN_COLAB = "google.colab" in sys.modules
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUT, exist_ok=True)

# locate or clone the repo
if os.path.isdir("ACSolverX/.git"):
    REPO = os.path.abspath("ACSolverX")
elif os.path.exists("experiments/stable_ac/baseline_n2/greedy_ac.py"):
    REPO = os.getcwd()                          # already inside the repo
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "ACSolverX"], check=True)
    REPO = os.path.abspath("ACSolverX")
print("repo:", REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numba", "numpy"], check=True)
sys.path.insert(0, os.path.join(REPO, "experiments/stable_ac/baseline_n2"))

In [ ]:
# --- import the verbatim solver + load the 6 ms640_solved tail presentations ---
import ast
from greedy_ac import solve_one           # numba-backed; runs ACRelatorSolver + verify_path

data_path = os.path.join(REPO, "data", "ms640_solved.txt")
lines = open(data_path).read().splitlines()
assert len(lines) == 640, len(lines)

PRES = {li: ast.literal_eval(lines[li]) for li in LINE_IDXS}

OUT_DIR = DRIVE_OUT if (USE_DRIVE and IN_COLAB) else os.path.join(REPO, "results/baseline_greedy/stragglers")
os.makedirs(OUT_DIR, exist_ok=True)
OUT = os.path.join(OUT_DIR, OUT_NAME)
print("ms640_solved tail lines (1-based):", [li + 1 for li in LINE_IDXS])
for li in LINE_IDXS:
    ms1190_idx, ms = TAIL6[li]
    print(f"  line {li + 1:3d}  1190MS idx {ms1190_idx:4d}  {ms}")
print("stream ->", OUT)

In [ ]:
# --- base-case gate: ms640 line 1 must solve + verify (warms numba pre-fork) --
bc = solve_one(ast.literal_eval(lines[0]), cap_mode=CAP, max_len=MAX_LEN, max_nodes=100_000)
print("base case ms640 line 1:", bc)
assert bc["solved"] and bc["path_verified"], "base case FAILED - do not trust the sweep"
print("OK base case passed; numba is warm (forked children inherit the compiled cache)")

In [ ]:
# --- run the 6 @ 1M: resumable, fresh forked child per line (peak RSS released) --
import resource, multiprocessing as mp

def jsonl_done(path):
    done = set()
    if os.path.exists(path):
        for line in open(path):
            line = line.strip()
            if not line:
                continue
            try:
                done.add(json.loads(line)["ms640_line"])
            except Exception:
                pass                             # ignore a torn trailing line; that line re-runs
    return done

def worker(line_idx):
    from greedy_ac import solve_one              # re-import inside the forked child
    r = solve_one(PRES[line_idx], cap_mode=CAP, max_len=MAX_LEN, max_nodes=BUDGET)
    ms1190_idx, ms = TAIL6[line_idx]
    # Linux (Colab): ru_maxrss is in KB
    r.update(
        ms640_line=line_idx + 1,
        ms1190_idx=ms1190_idx,
        ms=ms,
        budget_nodes=BUDGET,
        max_len=MAX_LEN,
        cap=CAP,
        peak_rss_gb=round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 / 1024, 2),
    )
    return r

done = jsonl_done(OUT)
todo = [li for li in LINE_IDXS if (li + 1) not in done]
print(f"{len(done)} already done (1-based lines) {sorted(done)};  running lines {[li+1 for li in todo]}")

ctx = mp.get_context("fork")                     # Colab is Linux/fork -> notebook worker is safe
with open(OUT, "a") as f:
    with ctx.Pool(processes=WORKERS, maxtasksperchild=1) as pool:   # fresh child per line
        for r in pool.imap_unordered(worker, todo):
            f.write(json.dumps(r) + "\n"); f.flush(); os.fsync(f.fileno())
            tag = "*** SOLVED ***" if r["solved"] and r["path_verified"] else "unsolved"
            print(f"  ms640 line {r['ms640_line']} ({r['ms']}, 1190MS idx {r['ms1190_idx']}): {tag}  "
                  f"nodes={r['nodes_explored']:,}  path_len={r['path_len']}  "
                  f"peak={r['peak_rss_gb']} GB  {r['wall_time_s']/60:.1f} min")
print("done ->", OUT)

In [ ]:
# --- summary ---------------------------------------------------------------
rows = {json.loads(l)["ms640_line"]: json.loads(l) for l in open(OUT) if l.strip()}
target_lines = [li + 1 for li in LINE_IDXS]
solved = [ln for ln in target_lines if rows.get(ln, {}).get("solved") and rows[ln].get("path_verified")]
unsolved = [ln for ln in target_lines if ln not in solved]

print(f"ms640 tail solved AND verified @ {BUDGET:,} nodes (sum cap L={MAX_LEN}):  {len(solved)}/6  lines {solved}")
print(f"still unsolved:  lines {unsolved}")
print(f"=> paper-trivial ms640 count = 634 (lines 1-634) + {len(solved)} here = {634 + len(solved)}  (Table 1: 640 @ 1M)")
print()
for ln in target_lines:
    r = rows.get(ln, {})
    print(f"  line {ln}: ms1190_idx={r.get('ms1190_idx')} {r.get('ms')}  "
          f"solved={r.get('solved')} verified={r.get('path_verified')}  "
          f"nodes={r.get('nodes_explored')} path_len={r.get('path_len')}  "
          f"max_len_along_path={r.get('max_len_along_path')} peak={r.get('peak_rss_gb')}GB")